# UPAR 2027 Track 2 — pipeline CSAR de bout en bout

Lance tout le pipeline depuis un seul endroit : données → étude des scores → backbone → entraînement → évaluation/calibration → export → test de la soumission comme Codabench.

* `MODE = "smoke"` : données synthétiques (labels réels, images aléatoires), CPU, ~1 min. Vérifie que tout s'enchaîne.
* `MODE = "full"` : vraies images (`python download_datasets.py` au préalable), GPU recommandé.

Toute la logique vit dans `src/` ; le notebook ne fait qu'appeler les mêmes points d'entrée que la ligne de commande (`python -m src.train`, etc.).

In [ ]:
import os
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np
import torch

# ---- paramètres ---------------------------------------------------------------
MODE = "smoke"            # "smoke" | "full"
BACKBONE = "resnet18" if MODE == "smoke" else "convnext_base"   # voir src/model.py: BACKBONES
HOLDOUT = None            # None (modèle final) | "Market1501" | "PA100k" | "PETA" (pli LODO)
EPOCHS = 1 if MODE == "smoke" else 20
BATCH_SIZE = 16 if MODE == "smoke" else 64
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORKERS = 0 if MODE == "smoke" else 8
RUN_DIR = ROOT / "runs" / f"{MODE}_{BACKBONE}{'_' + HOLDOUT if HOLDOUT else ''}"
CHECKPOINT = RUN_DIR / "model.pt"
print(f"mode={MODE} backbone={BACKBONE} holdout={HOLDOUT} device={DEVICE} -> {RUN_DIR}")

## 1. Données

In [ ]:
from src.data import load_split
from src.synthetic import make_synthetic_data

if MODE == "smoke":
    DATA_DIR = make_synthetic_data(ROOT / "data", RUN_DIR / "data", n_train=96, n_val=48)
else:
    DATA_DIR = ROOT / "data"

for name in ("train", "val"):
    split = load_split(DATA_DIR, name)
    domains, counts = np.unique(split.domains, return_counts=True)
    missing = sum(not (DATA_DIR / p).is_file() for p in split.images[:200])
    print(f"{name}: {len(split)} images, {len(split.queries)} requêtes, "
          f"{dict(zip(domains, counts.tolist()))}, images manquantes (200 premières): {missing}")

## 2. Étude des fonctions de score (sans images)

Vérité terrain de la validation officielle + bruit gaussien sur les logits : compare `l1` (exemple officiel), `loglik` (S2), `structured` (S3) et `mix` (E[ndom] + λ·P_exact).

In [ ]:
from src import evaluate

evaluate.main(["--data-dir", str(ROOT / "data"), "--simulate", "3.0",
               "--scores", "l1", "loglik", "structured", "mix", "--lams", "1", "30",
               "--max-queries", "300", "--no-domains"])

## 3. Backbone : poids ImageNet et débit

Charge les poids pré-entraînés et estime le temps d'inférence d'une galerie de 28 000 images (taille du test 2024) sur le matériel courant.

In [ ]:
from src import benchmark

benchmark.main(["--backbones", BACKBONE, "--device", DEVICE, "--batch-size", "8", "--iters", "1"])

## 4. Entraînement

In [ ]:
from src import train

train_args = ["--data-dir", str(DATA_DIR), "--out", str(RUN_DIR), "--backbone", BACKBONE,
              "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
              "--workers", str(WORKERS), "--device", DEVICE]
if HOLDOUT:
    train_args += ["--holdout", HOLDOUT]
if MODE == "smoke":
    train_args += ["--no-pretrained", "--height", "128", "--width", "64"]
train.main(train_args)

## 5. Évaluation, calibration et choix du score

En LODO, la calibration est refaite sur une moitié des requêtes du domaine exclu et la mesure sur l'autre moitié (`--calib-frac 0.5`). `--save-best` écrit le meilleur score et la calibration dans le checkpoint.

In [ ]:
eval_args = ["--data-dir", str(DATA_DIR), "--checkpoint", str(CHECKPOINT),
             "--scores", "loglik", "structured", "mix", "--lams", "1", "30", "--gammas", "0", "0.5",
             "--device", DEVICE, "--workers", str(WORKERS), "--save-best"]
if HOLDOUT:
    eval_args += ["--holdout", HOLDOUT, "--calib-frac", "0.5"]
if MODE == "smoke":
    eval_args += ["--no-domains"]
evaluate.main(eval_args)

## 6. Export de la soumission Codabench

In [ ]:
from src import export

archive = export.export(CHECKPOINT, RUN_DIR / "submission", overwrite=True)
print(archive, f"{archive.stat().st_size / 2**20:.1f} MiB")

## 7. Test de la soumission comme le programme d'ingestion

Importe le `run.py` exporté (pas le code du dépôt) et appelle `rank_gallery` sur une galerie de validation, puis calcule les métriques officielles.

In [ ]:
import importlib.util

from src.attributes import ATTRIBUTE_NAMES
from src.data import query_mask
from src.metrics import evaluate_output

spec = importlib.util.spec_from_file_location("submission_run", RUN_DIR / "submission" / "run.py")
submission = importlib.util.module_from_spec(spec)
spec.loader.exec_module(submission)
submission.WORKERS = WORKERS

gallery = load_split(DATA_DIR, "val")
if HOLDOUT:
    gallery = gallery.subset(gallery.domains == HOLDOUT)
if len(gallery.queries) > 367:   # taille du test 2024
    gallery = gallery.subset(query_mask(gallery, num_queries=367))
sample = {"attribute_names": list(ATTRIBUTE_NAMES),
          "queries": gallery.queries.tolist(),
          "gallery": [{"image_path": str(DATA_DIR / p)} for p in gallery.images]}

t0 = time.time()
output = submission.rank_gallery(sample)
print(f"{len(gallery.queries)} requêtes x {len(gallery)} images en {time.time() - t0:.1f}s")
evaluate_output(output, gallery.queries, gallery.labels)

## Suite

1. Lancer les trois plis LODO (`HOLDOUT = "Market1501" / "PA100k" / "PETA"`) et comparer les scores pli par pli.
2. Modèle final : `HOLDOUT = None`, puis soumettre `runs/full_<backbone>/submission.zip`.
3. Le mode transductif (`--transductive`) reste désactivé tant que les organisateurs ne l'ont pas autorisé par écrit.